In [1]:
#!/usr/bin/env python
"""
Test script to verify that pre-computed fingerprints work correctly
and provide performance benefits.
"""

import time
from rdkit import Chem
from oracle import (
    compute_max_similarity,
    precompute_reference_fingerprints,
    combined_rnamigos2_similarity_oracle
)

def test_fingerprint_caching():
    """Test that pre-computed fingerprints give the same results as on-the-fly computation."""
    
    # Sample reference molecules
    reference_smiles = [
        'CCO',  # Ethanol
        'CC(C)O',  # Isopropanol
        'c1ccccc1',  # Benzene
        'CC(=O)O',  # Acetic acid
        'CCN(CC)CC',  # Triethylamine
    ]
    
    # Test molecule
    test_smi = 'CCCO'  # Propanol
    
    print("Testing fingerprint caching...")
    print(f"Reference molecules: {len(reference_smiles)}")
    print(f"Test molecule: {test_smi}")
    print()
    
    # Method 1: On-the-fly fingerprint computation (slower)
    print("Method 1: Computing fingerprints on-the-fly...")
    start_time = time.time()
    similarity_1 = compute_max_similarity(
        test_smi,
        reference_smiles=reference_smiles
    )
    time_1 = time.time() - start_time
    print(f"  Result: {similarity_1:.4f}")
    print(f"  Time: {time_1*1000:.2f} ms")
    print()
    
    # Method 2: Pre-computed fingerprints (faster)
    print("Method 2: Using pre-computed fingerprints...")
    start_time = time.time()
    reference_fps = precompute_reference_fingerprints(reference_smiles)
    precompute_time = time.time() - start_time
    print(f"  Pre-computation time: {precompute_time*1000:.2f} ms")
    
    start_time = time.time()
    similarity_2 = compute_max_similarity(
        test_smi,
        reference_fingerprints=reference_fps
    )
    time_2 = time.time() - start_time
    print(f"  Result: {similarity_2:.4f}")
    print(f"  Query time: {time_2*1000:.2f} ms")
    print()
    
    # Verify results match
    print("Verification:")
    if abs(similarity_1 - similarity_2) < 1e-10:
        print("  ✓ Results match perfectly!")
    else:
        print(f"  ✗ Results differ: {abs(similarity_1 - similarity_2)}")
    print()
    
    # Performance comparison (simulate multiple queries)
    n_queries = 100
    print(f"Performance test with {n_queries} queries:")
    
    # Without caching
    start_time = time.time()
    for _ in range(n_queries):
        _ = compute_max_similarity(test_smi, reference_smiles=reference_smiles)
    time_no_cache = time.time() - start_time
    
    # With caching
    reference_fps = precompute_reference_fingerprints(reference_smiles)
    start_time = time.time()
    for _ in range(n_queries):
        _ = compute_max_similarity(test_smi, reference_fingerprints=reference_fps)
    time_with_cache = time.time() - start_time
    
    print(f"  Without caching: {time_no_cache*1000:.2f} ms ({time_no_cache/n_queries*1000:.2f} ms/query)")
    print(f"  With caching:    {time_with_cache*1000:.2f} ms ({time_with_cache/n_queries*1000:.2f} ms/query)")
    print(f"  Speedup: {time_no_cache/time_with_cache:.2f}x faster")
    print()
    
    print("Test completed successfully! ✓")

if __name__ == '__main__':
    test_fingerprint_caching()


/home/wangx86/miniconda3/envs/mol-opt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Testing fingerprint caching...
Reference molecules: 5
Test molecule: CCCO

Method 1: Computing fingerprints on-the-fly...
  Result: 0.5556
  Time: 0.30 ms

Method 2: Using pre-computed fingerprints...
  Pre-computation time: 0.08 ms
  Result: 0.5556
  Query time: 0.02 ms

Verification:
  ✓ Results match perfectly!

Performance test with 100 queries:
  Without caching: 8.20 ms (0.08 ms/query)
  With caching:    1.39 ms (0.01 ms/query)
  Speedup: 5.91x faster

Test completed successfully! ✓
